In [ ]:
# Notebook overview: 
# This notebook builds the module of wrapper class Simulator used in simulator_final. 
# This module executes the final simulator with a given theta initialization (theta_init).

import numpy as np
from simulator_final import ShotGradientEstimator, HybridModel, SimpleAdam, Trainer

class Simulator:
    def __init__(self, circuit, noise_model, H_cost, H_head,
                 classical_pre, data_loader, lambda0,
                 shots=128, lr=1e-2):
        self.grad_estimator = ShotGradientEstimator(
            circuit=circuit,
            noise_model=noise_model,
            H=H_cost,
            shots=shots,
            lambda0=lambda0,
        )
        self.hybrid_model = HybridModel(
            classical_pre=classical_pre,
            circuit=circuit,
            H_head=H_head,
        )
        self.optimizer = SimpleAdam(lr=lr)
        self.trainer = Trainer(
            hybrid_model=self.hybrid_model,
            grad_estimator=self.grad_estimator,
            optimizer=self.optimizer,
            data_loader=data_loader,
        )

    def run(self, theta_init, n_steps):
        theta = theta_init.copy()
        history = []
        for _ in range(n_steps):
            theta, loss = self.trainer.step(theta)
            history.append(loss)
        return theta, np.array(history)
    
# For example usage, see simulator_api.ipynb